# ML Algorithms Lab Practical — Logistic Regression Classification

**First classifier in the series.** This notebook follows the uploaded *Machine Learning Algorithms Cheatsheet* and includes preprocessing, hyperparameter tuning, cross-validation, extensive classification metrics, visualizations, and statistical significance testing.

## Logistic Regression hyperparameters from the cheat sheet

The cheat sheet (page 1) names these parameters: `penalty`, `C`, `solver`, `l1_ratio`, and `random_state`. It does **not** specify exact value ranges for Logistic Regression, so the grid below clearly labels the chosen ranges as practical additions rather than claiming they were printed in the cheat sheet.

- `penalty`: `l1`, `l2`, `elasticnet`
- `C`: `0.1`, `1`, `10`
- `solver`: chosen only from compatible solvers for each penalty
- `l1_ratio`: `0.25`, `0.5`, `0.75` for `elasticnet` only
- `random_state`: fixed at `42` for reproducibility
- `max_iter`: fixed at `5000` to help convergence; it is not listed in the cheat-sheet grid.

The grid is a **list of conditional parameter grids** so incompatible `penalty`/`solver` combinations are not evaluated. The notebook uses macro-F1 for tuning (useful when classes are imbalanced), then reports multiple metrics.

## How to use
1. Put your dataset CSV in the same folder as this notebook, or enter its path when prompted.
2. Set `TARGET` to the exact name of your classification target column.
3. Run the notebook from top to bottom.
4. Review the printed best hyperparameters, metrics, plots, and significance-test interpretation.

**Note:** The experiment compares Logistic Regression with and without PCA using identical outer folds. PCA is applied only to numeric features, while categorical one-hot features pass through. Results are dataset-dependent; this notebook has not been run on your personal dataset.

In [ ]:
# ============================================================
# 0. IMPORTS AND REPRODUCIBILITY
# ============================================================
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel, wilcoxon

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, matthews_corrcoef,
    cohen_kappa_score, log_loss, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve,
    auc
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
    cross_validate, learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme()

print("Imports complete. Random state:", RANDOM_STATE)


# ============================================================
# 1. LOAD DATA
# ============================================================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Found one CSV:", default_path)
    use_default = input("Use this file? [Y/n]: ").strip().lower()
    file_path = default_path if use_default in ("", "y", "yes") else input("Enter CSV path: ").strip()
elif len(csv_files) > 1:
    print("CSV files in current folder:")
    for i, f in enumerate(csv_files, 1):
        print(f"{i}. {f}")
    file_path = input("Enter the CSV filename/path to use: ").strip()
else:
    file_path = input("Enter the CSV file path: ").strip()

df = pd.read_csv(file_path)
print("\nDataset shape:", df.shape)
display(df.head())
print("\nData types and non-null counts:")
df.info()
print("\nDescriptive summary:")
display(df.describe(include="all").T)
print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nColumn names:", df.columns.tolist())


# ============================================================
# 2. SELECT TARGET AND BASIC DATA CLEANING
# ============================================================
print("\nColumns available for target selection:")
for col in df.columns:
    print(" -", col)

TARGET = input("\nEnter the exact classification target column name: ").strip()
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' was not found. Check spelling and capitalization.")

df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[df[TARGET].notna()].copy()
y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

# Drop likely identifier columns (including common ID names and nearly unique identifiers).
id_like = []
for col in X.columns:
    name = str(col).lower()
    unique_ratio = X[col].nunique(dropna=True) / max(len(X), 1)
    if name in {"id", "record_id", "row_id", "customer_id", "employee_id", "sample_id"}:
        id_like.append(col)
    elif name.endswith("_id") and unique_ratio > 0.80:
        id_like.append(col)
    elif unique_ratio > 0.98 and X[col].nunique(dropna=True) > 20:
        id_like.append(col)

if id_like:
    print("Dropping likely identifier columns:", id_like)
    X = X.drop(columns=id_like)

# Ensure classification target is discrete; preserve string labels if present.
if y.nunique(dropna=True) < 2:
    raise ValueError("Classification requires at least two target classes.")
if y.nunique(dropna=True) > 50 and y.dtype.kind in "if":
    print("WARNING: Target has more than 50 numeric values. Confirm that TARGET is categorical, not a regression target.")

print("\nTarget class counts:")
display(y.value_counts(dropna=False).to_frame("count"))
print("\nTarget class proportions:")
display(y.value_counts(normalize=True, dropna=False).to_frame("proportion"))

plt.figure(figsize=(8, 4))
order = y.astype(str).value_counts().index
sns.countplot(x=y.astype(str), order=order)
plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Numeric/categorical EDA. Plot only a manageable number of columns.
numeric_cols_eda = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols_eda = X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric features:", numeric_cols_eda)
print("Categorical features:", categorical_cols_eda)

if numeric_cols_eda:
    X[numeric_cols_eda].hist(figsize=(14, 9), bins=25)
    plt.suptitle("Numeric Feature Histograms")
    plt.tight_layout()
    plt.show()

    if len(numeric_cols_eda) >= 2:
        plt.figure(figsize=(10, 7))
        sns.heatmap(X[numeric_cols_eda].corr(), annot=False, cmap="coolwarm", center=0)
        plt.title("Numeric Feature Correlation Heatmap")
        plt.tight_layout()
        plt.show()

for col in categorical_cols_eda[:8]:
    plt.figure(figsize=(8, 3))
    X[col].astype("str").value_counts().head(15).plot(kind="bar")
    plt.title(f"Top categories: {col}")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


# ============================================================
# 3. TRAIN/TEST SPLIT (TEST SET HELD OUT FROM TUNING)
# ============================================================
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError("At least one target class has fewer than 2 rows. Stratified splitting/CV is not reliable; collect more data or revise the split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Training class proportions:")
display(y_train.value_counts(normalize=True).to_frame("train_proportion"))
print("Test class proportions:")
display(y_test.value_counts(normalize=True).to_frame("test_proportion"))


# ============================================================
# 4. PREPROCESSING: NUMERIC IMPUTATION + SCALING, CATEGORICAL IMPUTATION + OHE
#    All preprocessing is inside the pipeline to avoid leakage.
# ============================================================
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_one_hot_encoder():
    # Compatible with both newer and older scikit-learn versions.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_one_hot_encoder())
])

transformers = []
if num_cols:
    transformers.append(("num", numeric_pipe, num_cols))
if cat_cols:
    transformers.append(("cat", categorical_pipe, cat_cols))
if not transformers:
    raise ValueError("No usable predictor columns remain after removing the target/IDs.")

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
# PCA variant: PCA is applied only to the scaled numeric branch; categorical columns are one-hot encoded and passed through.
pca_transformers = []
if len(num_cols) >= 2:
    pca_numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE))
    ])
    pca_transformers.append(("num", pca_numeric_pipe, num_cols))
elif num_cols:
    # PCA needs at least two numeric features; for a one-numeric-feature dataset,
    # keep that branch imputed/scaled so the notebook can still run.
    pca_numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    pca_transformers.append(("num", pca_numeric_pipe, num_cols))
if cat_cols:
    pca_transformers.append(("cat", categorical_pipe, cat_cols))
pca_preprocessor = ColumnTransformer(transformers=pca_transformers, remainder="drop")

print("Numeric feature count:", len(num_cols))
print("Categorical feature count:", len(cat_cols))


# ============================================================
# 5. LOGISTIC REGRESSION + ALL CHEAT-SHEET PARAMETERS IN TUNING
# ============================================================
# From the cheat sheet: penalty, C, solver, l1_ratio, random_state.
# The sheet does not give Logistic Regression grid values; the C/l1_ratio
# ranges below are practical additions and are explicitly shown in the intro.
# Conditional grids avoid invalid solver/penalty combinations.
common = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}
param_grid = [
    # L1 penalty supports liblinear and saga.
    {
        **common,
        "model__penalty": ["l1"],
        "model__solver": ["liblinear", "saga"]
    },
    # L2 penalty supports liblinear, lbfgs, and saga.
    {
        **common,
        "model__penalty": ["l2"],
        "model__solver": ["liblinear", "lbfgs", "saga"]
    },
    # Elastic-net is supported by saga; tune l1_ratio only here.
    {
        **common,
        "model__penalty": ["elasticnet"],
        "model__solver": ["saga"],
        "model__l1_ratio": [0.25, 0.5, 0.75]
    }
]
print("Conditional parameter grids prepared.")
print("Parameters tuned: penalty, C, solver, l1_ratio (elasticnet only), class_weight (additional imbalance option).")
print("Fixed for reproducibility/stability: random_state=42, max_iter=5000.")


# ============================================================
# 6. NESTED STRATIFIED CV: PCA VS NO PCA, IDENTICAL OUTER FOLDS
# ============================================================
# Outer CV estimates generalization; inner GridSearchCV chooses parameters.
# The untouched test set is not used during this comparison.
min_class_train = int(y_train.value_counts().min())
outer_splits = min(5, min_class_train)
if outer_splits < 2:
    raise ValueError("Insufficient examples per class for at least 2-fold stratified CV.")
inner_splits = min(3, min_class_train)
if inner_splits < 2:
    raise ValueError("Insufficient examples per class for inner CV.")

outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
fold_rows = []
fold_predictions = {}

for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    inner_min = int(y_tr.value_counts().min())
    cv_inner = StratifiedKFold(n_splits=min(inner_splits, inner_min), shuffle=True, random_state=RANDOM_STATE)
    if cv_inner.n_splits < 2:
        print(f"Skipping fold {fold}: too few examples in a class for inner CV.")
        continue

    for representation, prep in [("Without PCA", preprocessor), ("With PCA", pca_preprocessor)]:
        pipe = Pipeline([
            ("preprocess", clone(prep)),
            ("model", LogisticRegression(random_state=RANDOM_STATE, max_iter=5000))
        ])
        search = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="f1_macro",
            cv=cv_inner,
            n_jobs=-1,
            refit=True,
            return_train_score=True,
            error_score="raise"
        )
        start = time.time()
        search.fit(X_tr, y_tr)
        elapsed = time.time() - start
        pred_train = search.predict(X_tr)
        pred_valid = search.predict(X_va)

        row = {
            "fold": fold,
            "representation": representation,
            "best_inner_cv_macro_f1": search.best_score_,
            "accuracy": accuracy_score(y_va, pred_valid),
            "balanced_accuracy": balanced_accuracy_score(y_va, pred_valid),
            "precision_macro": precision_score(y_va, pred_valid, average="macro", zero_division=0),
            "recall_macro": recall_score(y_va, pred_valid, average="macro", zero_division=0),
            "f1_macro": f1_score(y_va, pred_valid, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_va, pred_valid, average="weighted", zero_division=0),
            "train_macro_f1": f1_score(y_tr, pred_train, average="macro", zero_division=0),
            "fit_seconds": elapsed,
            "best_params": search.best_params_
        }
        fold_rows.append(row)
        fold_predictions[(fold, representation)] = {
            "y_true": y_va.copy(),
            "y_pred": pd.Series(pred_valid, index=y_va.index),
            "model": search.best_estimator_,
            "X_valid": X_va.copy()
        }
        print(f"\nFold {fold} | {representation}")
        print("Best parameters:", search.best_params_)
        print(f"Best inner-CV macro F1: {search.best_score_:.4f}")
        print(f"Validation macro F1: {row['f1_macro']:.4f} | Accuracy: {row['accuracy']:.4f} | Time: {elapsed:.1f}s")

cv_results = pd.DataFrame(fold_rows)
display(cv_results)

if cv_results.empty:
    raise RuntimeError("No CV folds completed. Check target class counts and dataset size.")

metric_cols = ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro",
               "f1_macro", "f1_weighted", "best_inner_cv_macro_f1", "fit_seconds"]
print("\nCross-validation summary (mean and standard deviation):")
display(cv_results.groupby("representation")[metric_cols].agg(["mean", "std"]))


# ============================================================
# 7. STATISTICAL SIGNIFICANCE: PAIRED T-TEST + WILCOXON
# ============================================================
# Same outer folds make these paired comparisons appropriate.
wide = cv_results.pivot(index="fold", columns="representation", values="f1_macro").dropna()
if len(wide) >= 2 and {"Without PCA", "With PCA"}.issubset(wide.columns):
    without = wide["Without PCA"].to_numpy()
    with_pca = wide["With PCA"].to_numpy()
    t_stat, t_p = ttest_rel(without, with_pca)
    try:
        w_stat, w_p = wilcoxon(without, with_pca, zero_method="wilcox", alternative="two-sided")
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    print("Paired comparison: Without PCA vs With PCA, using outer-fold macro F1")
    print(f"Paired t-test: statistic={t_stat:.4f}, p-value={t_p:.6f}")
    print(f"Wilcoxon signed-rank: statistic={w_stat}, p-value={w_p}")
    print("Significance level: 0.05")
    print("Interpretation:", "statistically significant fold-score difference" if t_p < 0.05 else "no statistically significant fold-score difference detected")
    print("Caution: only a small number of outer folds are available, so treat significance tests as supportive rather than definitive.")
else:
    print("Not enough matched fold results to perform the PCA vs no-PCA significance tests.")


# ============================================================
# 8. PLOTS: CV SCORES AND CONFUSION MATRICES
# ============================================================
plt.figure(figsize=(8, 4))
sns.boxplot(data=cv_results, x="representation", y="f1_macro")
sns.stripplot(data=cv_results, x="representation", y="f1_macro", color="black", alpha=0.6)
plt.title("Outer-fold Macro F1: PCA vs No PCA")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.lineplot(data=cv_results, x="fold", y="accuracy", hue="representation", marker="o")
plt.title("Outer-fold Accuracy")
plt.tight_layout()
plt.show()

for representation in ["Without PCA", "With PCA"]:
    available_folds = [f for f in range(1, outer_splits + 1) if (f, representation) in fold_predictions]
    if available_folds:
        item = fold_predictions[(available_folds[-1], representation)]
        cm = confusion_matrix(item["y_true"], item["y_pred"], labels=np.unique(y_train))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_train))
        disp.plot(xticks_rotation=45, cmap="Blues", values_format="d")
        plt.title(f"Fold {available_folds[-1]} Confusion Matrix — {representation}")
        plt.tight_layout()
        plt.show()


# ============================================================
# 9. FINAL MODEL: TUNE ON TRAINING DATA, EVALUATE ON HELD-OUT TEST DATA
# ============================================================
# Select PCA/no-PCA using mean outer-fold macro F1, then tune on all training data.
mean_scores = cv_results.groupby("representation")["f1_macro"].mean().sort_values(ascending=False)
selected_representation = mean_scores.index[0]
selected_prep = preprocessor if selected_representation == "Without PCA" else pca_preprocessor
print("Representation selected from outer CV:", selected_representation)

final_cv_splits = min(5, int(y_train.value_counts().min()))
final_cv = StratifiedKFold(n_splits=final_cv_splits, shuffle=True, random_state=RANDOM_STATE)
final_pipe = Pipeline([
    ("preprocess", clone(selected_prep)),
    ("model", LogisticRegression(random_state=RANDOM_STATE, max_iter=5000))
])
final_search = GridSearchCV(
    final_pipe, param_grid=param_grid, scoring="f1_macro",
    cv=final_cv, n_jobs=-1, refit=True, return_train_score=True, error_score="raise"
)
final_start = time.time()
final_search.fit(X_train, y_train)
final_seconds = time.time() - final_start
best_model = final_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nFINAL SELECTED MODEL")
print("Representation:", selected_representation)
print("Best hyperparameters:", final_search.best_params_)
print(f"Best training CV macro F1: {final_search.best_score_:.4f}")
print(f"Final tuning + fit time: {final_seconds:.2f} seconds")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Comprehensive hold-out metrics.
test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "matthews_corrcoef": matthews_corrcoef(y_test, y_pred),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred)
}
try:
    if hasattr(best_model, "predict_proba"):
        proba = best_model.predict_proba(X_test)
        test_metrics["log_loss"] = log_loss(y_test, proba, labels=best_model.classes_)
        if len(best_model.classes_) == 2:
            test_metrics["roc_auc"] = roc_auc_score(y_test, proba[:, 1])
            test_metrics["average_precision"] = average_precision_score(
                (y_test == best_model.classes_[1]).astype(int), proba[:, 1]
            )
        else:
            test_metrics["roc_auc_ovr_macro"] = roc_auc_score(
                y_test, proba, multi_class="ovr", average="macro", labels=best_model.classes_
            )
except Exception as exc:
    print("Some probability-based metrics were skipped:", exc)

print("\nHold-out test metrics:")
display(pd.DataFrame({"metric": list(test_metrics.keys()), "value": list(test_metrics.values())}).sort_values("metric"))


# ============================================================
# 10. TEST-SET CONFUSION MATRIX, ROC AND PRECISION-RECALL CURVES
# ============================================================
labels = best_model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=45, cmap="Blues", values_format="d")
plt.title("Final Test-Set Confusion Matrix")
plt.tight_layout()
plt.show()

if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    if len(labels) == 2:
        positive_label = labels[1]
        y_binary = (y_test == positive_label).astype(int)
        fpr, tpr, _ = roc_curve(y_binary, proba[:, 1])
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve (positive class = {positive_label})")
        plt.legend()
        plt.tight_layout()
        plt.show()

        precision_vals, recall_vals, _ = precision_recall_curve(y_binary, proba[:, 1])
        ap = average_precision_score(y_binary, proba[:, 1])
        plt.figure(figsize=(6, 5))
        plt.plot(recall_vals, precision_vals, label=f"Average Precision = {ap:.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        y_bin = label_binarize(y_test, classes=labels)
        plt.figure(figsize=(7, 5))
        for i, label in enumerate(labels):
            if i < proba.shape[1] and y_bin[:, i].sum() > 0 and y_bin[:, i].sum() < len(y_bin):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
                plt.plot(fpr, tpr, label=f"{label} (AUC={auc(fpr, tpr):.3f})")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Multiclass One-vs-Rest ROC Curves")
        plt.legend()
        plt.tight_layout()
        plt.show()


# ============================================================
# 11. LEARNING CURVE AND GRID SEARCH DIAGNOSTICS
# ============================================================
# Learning curve uses the selected pipeline structure; it is diagnostic,
# not used to choose the final hyperparameters.
try:
    lc_cv = StratifiedKFold(n_splits=min(5, int(y_train.value_counts().min())), shuffle=True, random_state=RANDOM_STATE)
    train_sizes, train_scores, val_scores = learning_curve(
        clone(best_model), X_train, y_train, cv=lc_cv,
        scoring="f1_macro", train_sizes=np.linspace(0.3, 1.0, 5),
        n_jobs=-1, shuffle=True, random_state=RANDOM_STATE
    )
    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training macro F1")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation macro F1")
    plt.xlabel("Training examples")
    plt.ylabel("Macro F1")
    plt.title("Learning Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Learning curve skipped:", exc)

cv_table = pd.DataFrame(final_search.cv_results_).sort_values("rank_test_score")
print("\nTop 10 hyperparameter combinations:")
display(cv_table[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10))
print("\nBest CV score:", final_search.best_score_)
print("Best parameters:", final_search.best_params_)


# ============================================================
# 12. FEATURE COEFFICIENTS (WHEN AVAILABLE)
# ============================================================
try:
    fitted_preprocessor = best_model.named_steps["preprocess"]
    fitted_lr = best_model.named_steps["model"]
    feature_names = fitted_preprocessor.get_feature_names_out()
    coef = fitted_lr.coef_
    if coef.shape[0] == 1:
        coef_series = pd.Series(coef[0], index=feature_names).sort_values()
        print("Most negative coefficients:")
        display(coef_series.head(15).to_frame("coefficient"))
        print("Most positive coefficients:")
        display(coef_series.tail(15).sort_values(ascending=False).to_frame("coefficient"))
        top_coef = pd.concat([coef_series.head(10), coef_series.tail(10)]).sort_values()
        plt.figure(figsize=(9, 6))
        top_coef.plot(kind="barh")
        plt.title("Logistic Regression Coefficients (binary classification)")
        plt.xlabel("Coefficient")
        plt.tight_layout()
        plt.show()
    else:
        # Multiclass: show top absolute coefficients for each class.
        for i, cls in enumerate(fitted_lr.classes_):
            coef_series = pd.Series(coef[i], index=feature_names)
            top = coef_series.reindex(coef_series.abs().sort_values(ascending=False).head(12).index)
            print(f"Top coefficients for class {cls}:")
            display(top.to_frame("coefficient"))
except Exception as exc:
    print("Coefficient plot skipped (for example, PCA transforms feature names):", exc)


# ============================================================
# 13. STATISTICAL SIGNIFICANCE ON FINAL TEST PREDICTIONS (CAUTION)
# ============================================================
print("Statistical comparison was performed on paired outer-fold scores for PCA vs no PCA.")
print("Do not treat the held-out test set as repeated independent experiments.")
print("For a rigorous comparison between two different classifiers, evaluate both on the same folds")
print("and compare paired fold-level scores (e.g., Wilcoxon signed-rank or paired t-test).")
